# PV-Prognose mit Wetterdaten

PV-Anlage mit Ost- und Westdach (Einstellungen in `solar_wetter.py`) und gemessenem Horizont. Ablauf:

1. **Messdaten:** SMA-Export aus `Daten\SMA Export`, bereinigt.
2. **Wetter:** stündliche Einstrahlung und Temperatur von Open-Meteo, rückwirkend und als Vorhersage.
3. **Modell (pvlib):**
   - Sonnenstand
   - Einstrahlung auf beide Dachflächen
   - Horizont blockiert die direkte Sonne
   - Zelltemperatur
4. **Kalibrierung:** Alle übrigen Verluste stecken in einem Faktor je Monat, angepasst an die gemessenen Tage.
5. **Prognose** für die nächsten 7 Tage.

Die Logik steckt in `solar_wetter.py`.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from solar_wetter import *

tage = sma_tage()
print(tage.flag.value_counts().rename({"ok": "gültig", "aus": "Anlage aus (2023, nicht angemeldet)", "korrigiert": "Logger-Ausfall korrigiert"}).to_string())
tage.loc["2025-07-06":"2025-07-11", ["erzeugung", "einspeisung", "flag"]].round(1)

## Kalibrierung: Modell mit echtem Wetter gegen die Messung

In [ ]:
wetter = wetter_archiv("2023-07-20", "2025-12-31")
k, vgl = kalibrieren(tage, modell(wetter))
g = guete(vgl)
print(f"{g['tage']} gültige Tage: R² = {g['r2']:.3f}, mittlerer Fehler {g['mae_kwh']:.2f} kWh/Tag ({g['mae_prozent']:.1f} %), "
      f"Abweichung über alles {g['bias_prozent']:+.1f} %")
print("Kalibrierfaktor je Monat (alle Verluste):", " ".join(f"{m}:{v:.2f}" for m, v in k.items()))

ok = vgl[vgl.flag == "ok"]
mon = ok.groupby(pd.DatetimeIndex(ok.index).strftime("%m.%y"), sort=False)[["gemessen", "modell"]].sum()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
mon.plot.bar(ax=a1, width=0.8, color=["#e8a33d", "#4a7ab5"])
a1.tick_params(axis="x", labelrotation=90)
a1.set(ylabel="kWh", title="Monatsertrag: Messung gegen Modell (nur gültige Tage)")
a2.scatter(ok.gemessen, ok.modell, s=6, alpha=0.5)
a2.plot([0, 50], [0, 50], "k--", lw=1)
a2.set(xlabel="gemessen kWh/Tag", ylabel="Modell kWh/Tag", title="Tageswerte")
plt.tight_layout(); plt.show()
ok.groupby(ok.index.year)[["gemessen", "modell"]].sum().round(0)

**Einordnung:**

- Die Tageswerte streuen um etwa 15 %. Die Wetterdaten sind Reanalyse-Werte für ein Raster von einigen Kilometern, einzelne Wolken über dem Dach sieht das Modell nicht.
- Monats- und Jahressummen treffen auf wenige Prozent, auch wenn man mit einem Jahr kalibriert und das nächste vorhersagt.
- Die Faktoren sind im Winter deutlich kleiner als im Sommer.
  - Ein Teil davon sind normale Verluste: Wechselrichter, Leitungen, Verschmutzung, Alterung.
  - Im Winter kommen Schnee, flache Sonne und Teile des Horizonts dazu, die nicht vermessen sind.

## Prognose für die nächsten Tage

In [ ]:
p = prognose(k, days=8)
jetzt = pd.Timestamp.now(tz="Europe/Berlin")
heute = jetzt.normalize()
rest = p[(p.index > jetzt) & (p.index < heute + pd.Timedelta(days=1))]["gesamt"].sum()
print(f"Heute ganzer Tag (Prognose): {p[(p.index > heute) & (p.index <= heute + pd.Timedelta(days=1))]['gesamt'].sum():.1f} kWh, davon ab jetzt noch {rest:.1f} kWh")
p = p[(p.index > heute + pd.Timedelta(days=1)) & (p.index <= heute + pd.Timedelta(days=8))]
tag = p.groupby((p.index - pd.Timedelta(minutes=1)).date)[["Ost", "West", "gesamt"]].sum().round(1)
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(p.index, p["gesamt"], step="pre", alpha=0.6, label="gesamt")
ax.step(p.index, p["Ost"], where="pre", lw=1, label="Ost")
ax.step(p.index, p["West"], where="pre", lw=1, label="West")
ax.set(ylabel="kWh pro Stunde", title="Prognose stündlich (kalibriert)")
ax.legend(); plt.show()
tag.rename(columns={"gesamt": "gesamt kWh"})